In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets
from PIL import Image
import pickle
import bidsio
import sys
sys.path.append('../')
from helpers import *
from fns_grid import *
from bandlimited_signal import *


device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
seed = 0 
BIDS_LOADER = bidsio.BIDSLoader(data_entities=[{'subject': '',
                                              'session': '',
                                              'suffix': 'T1w',
                                              'space': 'MNI152NLin2009aSym'}],
                              target_entities=[],
                              data_derivatives_names=['ATLAS'],
                              batch_size=1,
                              root_dir='./atlas/data/test/')


In [ ]:
def load_gt(dataset, id_val, bandlimit = 0.6):
    if dataset=='div2k':
        image_pil = Image.open(f'./DIV2K/DIV2K_train_LR_x8/0{str(id_val)}x8.png')
        RES = 128
    elif dataset == 'sphere':
        image = SparseSphereSignal(dimension=2, length=128, bandlimit=bandlimit, seed=id_val, generate=False).signal
    elif dataset == 'bandlimited':
        image = BandlimitedSignal(dimension=2, length=128, bandlimit=bandlimit, seed=id_val, generate=False).signal
    elif dataset == 'mri':
        RES = 128
        tmp = BIDS_LOADER.load_sample(idx = id_val, data_only=True) / 255.0
        vol = resize(tmp, (1, RES, RES, RES))[0]
        slice_idx = 48
        image = vol[:, :, slice_idx]  # (RES, RES)
    else:
        raise ValueError(f"Dataset {dataset} not supported.")

    if dataset not in ['sphere', 'bandlimited', 'mri']:  
        image = np.array(image_pil.convert('L').resize((RES, RES))) / 255.0 
        
    return image


In [ ]:
dataset = 'mri'
if dataset in ['sphere', 'bandlimited']:
    image_idx = 1234
else:
    image_idx = 131
image = load_gt(dataset, image_idx)
learning_rate, iters = 5e-2, 1000

mask = None
if dataset == 'mri':
    mask = np.fft.fftshift(np.ones((128, 128))).astype(np.complex64)

model_sizes = [3300, 6600]
outputs = {}
to_save_outputs = {}
for model_size in model_sizes:
    r = compute_params_from_model_size("grid_eta", None, model_size)
    print(f'grid_reso: {r}')
    torch.manual_seed(seed)
    output = fit_grid(target_signal=image, r = r, iters = iters, lr=learning_rate, interp="bilinear", log_interval=250, seed=0, count_params=True)
    outputs[f'{r}'] = output     
    to_save_outputs[f'{r}'] = output['best_pred']
    error = np.linalg.norm(image.flatten() - output['best_pred'].flatten())                       
    print(f"Error: {error:.3e}, Loss: {output['best_loss']:.3e}")

with open(f"2d_{dataset}/grid.pkl", "wb") as f:
    pickle.dump(to_save_outputs, f)

In [ ]:
plot_error_heatmaps(image, outputs, model_name="Grid")